# Workbench

**The development surface for this project.** Every other notebook in this folder is
generated by a script in `src/` and rewritten on each build. This one is maintained by
hand, because it is where the work is done rather than where it is reported.

It covers the step that comes before any code is written: establishing what is actually
in the warehouse, so that the next join, quality check or reconciliation line is written
against the data rather than against an assumption about it.

The helpers live in [`src/explore.py`](../src/explore.py), so the same calls work from a
plain Python session or from any other notebook. The connection is **read-only** -
nothing here can alter the warehouse the reports depend on.

| Call | Answers |
|---|---|
| `w.tables()` / `w.tables("mart")` | what objects exist, and how big |
| `w.find("country")` | which tables hold a column with this name |
| `w.cols("core.fact_plan")` | columns, types, nulls, distinct values, an example |
| `w.peek("stg.plan_submissions", 10)` | the first rows |
| `w.values("stg.plan_submissions", "currency_code")` | value counts for one column |
| `w.grain("core.fact_plan", "canonical_country", "plan_month")` | is it unique at this grain |
| `w.duplicates(table, *keys)` | the offending rows when it is not |
| `w.overlap(left, lkey, right, rkey)` | do two join keys actually match |
| `w.owns("mart.price_volume_mix")` | which file and line builds it |
| `w.source("core.dim_market")` | the production SQL, printed from the file |
| `w.tie(label, sql_a, sql_b)` | does a total survive the next layer |
| `w.q("SELECT ...")` | anything else |


In [1]:
import sys

import pandas as pd

sys.path.insert(0, "../src")
from explore import open_warehouse

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 60)

w = open_warehouse()
print(w.path)

C:\Users\phanh\data_analysis\p_projects\the_look_ecommerce\new_analysis\artifacts\thelook_analytics.duckdb


---

## 1. What exists

The starting point when the object is known by concept rather than by name.


In [2]:
display(w.tables("mart"))

,schema,name,kind,rows
0,mart,acquisition_source_value,table,5
1,mart,actual_country_month,table,770
2,mart,cart_abandonment_segments,table,22
3,mart,channel_quality,table,5
4,mart,cohort_retention,table,1936
5,mart,commercial_scorecard,table,9
6,mart,customer_360,table,80095
7,mart,delivery_performance_dc,table,10
8,mart,executive_snapshot,table,12
9,mart,forecast_accuracy,table,3


In [3]:
# When you know the concept but not where it lives.
display(w.find("canonical_country"))

,schema,table,column,type
0,core,dim_country,canonical_country,VARCHAR
1,core,dim_market,canonical_country,VARCHAR
2,core,fact_plan,canonical_country,VARCHAR
3,mart,actual_country_month,canonical_country,VARCHAR
4,mart,forecast_country_month,canonical_country,VARCHAR
5,mart,plan_vs_actual_monthly,canonical_country,VARCHAR
6,raw,country_master,canonical_country,VARCHAR
7,stg,plan_submissions,canonical_country,VARCHAR


In [4]:
# Every layer, not just the one being worked on.
display(w.q("""
    SELECT table_schema AS layer, COUNT(*) AS objects
    FROM information_schema.tables
    WHERE table_schema IN ('raw','stg','core','mart','qa')
    GROUP BY 1
    ORDER BY CASE table_schema WHEN 'raw' THEN 1 WHEN 'stg' THEN 2
                               WHEN 'core' THEN 3 WHEN 'mart' THEN 4 ELSE 5 END
"""))

,layer,objects
0,raw,10
1,stg,9
2,core,14
3,mart,26
4,qa,6


The marts are the layer with the most objects, because there is one per business
question. Grouping them by the question they answer is the fastest way to see the
scope of the project - the commercial tables are one group of six, not the whole
of it.


In [5]:
DOMAINS = {
    "Acquisition and funnel": [
        "funnel_stage_channel", "funnel_monthly_channel", "channel_quality",
        "acquisition_source_value", "cart_abandonment_segments"],
    "Revenue, product and customer": [
        "sales_monthly", "customer_360", "cohort_retention",
        "product_performance_category", "product_performance_brand",
        "geography_performance_country"],
    "Operations and inventory": [
        "operations_monthly", "delivery_performance_dc",
        "return_risk_segments", "inventory_performance"],
    "Commercial planning and variance": [
        "actual_country_month", "plan_vs_actual_monthly", "plan_vs_actual_region",
        "price_volume_mix", "commercial_scorecard"],
    "Forecasting (next step)": [
        "forecast_monthly", "forecast_country_month", "forecast_accuracy",
        "forecast_vs_plan", "forecast_scenarios"],
    "Executive summary": ["executive_snapshot"],
}

survey = pd.DataFrame(
    [{"domain": domain, "mart": name,
      "rows": int(w.q(f"SELECT COUNT(*) AS n FROM mart.{name}").n.iloc[0])}
     for domain, names in DOMAINS.items() for name in names]
)
display(survey)

# The same anti-join idea, turned on the map itself: nothing unaccounted for.
everything = set(w.tables("mart").name)
mapped = {name for names in DOMAINS.values() for name in names}
print("marts not in the map :", sorted(everything - mapped) or "none")
print("map entries missing  :", sorted(mapped - everything) or "none")

,domain,mart,rows
0,Acquisition and funnel,funnel_stage_channel,20
1,Acquisition and funnel,funnel_monthly_channel,325
2,Acquisition and funnel,channel_quality,5
3,Acquisition and funnel,acquisition_source_value,5
4,Acquisition and funnel,cart_abandonment_segments,22
5,"Revenue, product and customer",sales_monthly,65
6,"Revenue, product and customer",customer_360,80095
7,"Revenue, product and customer",cohort_retention,1936
8,"Revenue, product and customer",product_performance_category,36
9,"Revenue, product and customer",product_performance_brand,2755


marts not in the map : none
map entries missing  : none


---

## 2. What is inside a table

`cols()` is the first call on any unfamiliar table. Three columns in its output carry
most of the information:

- **`nulls`** - a LEFT JOIN onto a column containing nulls produces nulls downstream
  that were never planned for
- **`distinct` against `rows`** - equal values mean the column is a candidate key
- **`type`** - a `VARCHAR` date will not join to a `DATE` one

The same information for **every** table at once, including what each column
means in context, is in [`docs/warehouse_catalogue.md`](../docs/warehouse_catalogue.md) -
65 objects and 674 columns, regenerated from the warehouse on every build.


In [6]:
display(w.cols("core.fact_plan"))

,column,type,rows,nulls,null_pct,distinct,example
0,canonical_country,VARCHAR,287,0,0.0,13,Australia
1,region,VARCHAR,287,0,0.0,3,AMER
2,sub_region,VARCHAR,287,0,0.0,10,Benelux
3,plan_month,DATE,287,0,0.0,24,2023-01-01
4,plan_date_key,INTEGER,287,0,0.0,24,20230101
5,currency_code,VARCHAR,287,0,0.0,10,AUD
6,budget_rate_to_usd,DOUBLE,287,0,0.0,10,0.00025
7,plan_net_sales_local,DOUBLE,287,0,0.0,250,0.0
8,plan_net_sales_usd,DOUBLE,287,0,0.0,251,0.0
9,plan_units,HUGEINT,287,0,0.0,231,1


In [7]:
display(w.peek("stg.plan_submissions", 8))

,submission_batch,submitted_country,submitted_country_trimmed,canonical_country,region,sub_region,plan_month,currency_code,budget_rate_to_usd,plan_net_sales_local,plan_net_sales_usd,plan_units,flag_unmapped_country,flag_missing_fx_rate,flag_blank_target,flag_negative_target,flag_whitespace_in_key
0,FY2023-PLAN-V1,Australia,Australia,Australia,APAC,Oceania,2023-01-01,AUD,0.66,5000.00,3300.0000,52,0,0,0,0,0
1,FY2023-PLAN-V1,Australia,Australia,Australia,APAC,Oceania,2023-02-01,AUD,0.66,1969.70,1300.0020,33,0,0,0,0,0
2,FY2023-PLAN-V1,Australia,Australia,Australia,APAC,Oceania,2023-03-01,AUD,0.66,3787.88,2500.0008,63,0,0,0,0,0
3,FY2023-PLAN-V1,Australia,Australia,Australia,APAC,Oceania,2023-04-01,AUD,0.66,7272.73,4800.0018,74,0,0,0,0,0
4,FY2023-PLAN-V1,Australia,Australia,Australia,APAC,Oceania,2023-05-01,AUD,0.66,6212.12,4099.9992,76,0,0,0,0,0
5,FY2023-PLAN-V1,Australia,Australia,Australia,APAC,Oceania,2023-06-01,AUD,0.66,6515.15,4299.9990,73,0,0,0,0,0
6,FY2023-PLAN-V1,Australia,Australia,Australia,APAC,Oceania,2023-07-01,AUD,0.66,7575.76,5000.0016,77,0,0,0,0,0
7,FY2023-PLAN-V1,Australia,Australia,Australia,APAC,Oceania,2023-08-01,AUD,0.66,6969.70,4600.0020,74,0,0,0,0,0


In [8]:
# Value counts are the fastest way to see a defect.
display(w.values("stg.plan_submissions", "currency_code", 12))

,value,rows,share
0,EUR,96,0.3288
1,GBP,24,0.0822
2,KRW,24,0.0822
3,JPY,24,0.0822
4,BRL,24,0.0822
5,CNY,24,0.0822
6,USD,24,0.0822
7,AUD,24,0.0822
8,PLN,21,0.0719
9,COP,7,0.0240


---

## 3. Tracing one row through every layer

The layers are easier to understand on a single row than in the abstract. Line item
**5** is followed below from the delivered CSV to the monthly mart, and each step
shows what that layer added and what it was not allowed to change.


In [9]:
print("RAW - the source file, landed exactly as delivered. No types, no renames.")
display(w.q("SELECT * FROM raw.order_items WHERE id = 5"))

RAW - the source file, landed exactly as delivered. No types, no renames.


,id,order_id,user_id,product_id,inventory_item_id,status,created_at,shipped_at,delivered_at,returned_at,sale_price
0,5,2,2,23805,11,Complete,2024-01-27 22:36:43,2024-01-25 00:18:00,2024-01-26 15:54:00,NaT,95.989998


In [10]:
print("STG - typed and renamed. Same row, same values, nothing dropped or derived.")
display(w.q("SELECT * FROM stg.order_items WHERE order_item_id = 5"))

STG - typed and renamed. Same row, same values, nothing dropped or derived.


,order_item_id,order_id,user_id,product_id,inventory_item_id,item_status,created_at,shipped_at,delivered_at,returned_at,sale_price
0,5,2,2,23805,11,Complete,2024-01-27 22:36:43,2024-01-25 00:18:00,2024-01-26 15:54:00,NaT,95.989998


In [11]:
print("CORE - joined to product, customer and date; the measures appear here.")
display(w.q("""
    SELECT order_item_id, order_id, customer_id, product_id, acquisition_source,
           item_status, gross_sales_value, net_sales_value, net_profit_value
    FROM core.fact_order_item WHERE order_item_id = 5
"""))

CORE - joined to product, customer and date; the measures appear here.


,order_item_id,order_id,customer_id,product_id,acquisition_source,item_status,gross_sales_value,net_sales_value,net_profit_value
0,5,2,2,23805,Organic,Complete,95.989998,95.989998,53.562419


In [12]:
print("MART - aggregated to the grain the question needs. The row is now one")
print("contribution inside a monthly total, which is why individual rows stop")
print("being visible at this layer.")
display(w.q("""
    SELECT month_start, items, gross_sales_value, net_sales_value, net_margin_rate
    FROM mart.sales_monthly
    WHERE month_start = (SELECT date_trunc('month', order_item_created_at)
                         FROM core.fact_order_item WHERE order_item_id = 5)
"""))

MART - aggregated to the grain the question needs. The row is now one
contribution inside a monthly total, which is why individual rows stop
being visible at this layer.


,month_start,items,gross_sales_value,net_sales_value,net_margin_rate
0,2024-01-01,7796,468036.380438,348737.740298,0.518917


| Layer | What it adds | What it may not do |
|---|---|---|
| `raw` | nothing - the file as delivered | be edited by hand |
| `stg` | types and column names | join, filter, derive, or drop a row |
| `core` | dimension keys, derived measures, status flags | change the row count |
| `mart` | aggregation to the grain the question needs | introduce a number not derivable from `core` |

Those four sentences are the layer contract, and they are what makes the
reconciliation in section 6 possible. If staging may not drop a row, then a row count
that changes between source and staging is a defect with exactly one place to look.


---

## 4. Before a JOIN

Two questions, answered in this order, before any join is written.

**Is the right-hand side unique at the key?** If it is not, the join multiplies rows and
the total silently grows. Nothing errors and no warning appears - the number is simply
larger than it should be.


In [13]:
display(w.grain("core.fact_plan", "canonical_country", "plan_month"))

,table,keys,rows,distinct_keys,duplicates,verdict
0,core.fact_plan,"canonical_country, plan_month",287,287,0,unique at this grain


In [14]:
# dim_country is a mapping table, so it is deliberately NOT unique on the market.
# This is why dim_market exists separately.
display(w.grain("core.dim_country", "canonical_country"))
display(w.duplicates("core.dim_country", "canonical_country", n=5))

,table,keys,rows,distinct_keys,duplicates,verdict
0,core.dim_country,canonical_country,16,14,2,NOT unique - a join on these keys will multiply rows


,canonical_country,rows
0,Spain,2
1,Germany,2


**Do the two keys actually match?** A join that matches nothing still returns rows, just
with NULLs where the measures should be. `overlap()` runs the anti-join in both
directions, which is the only way to see it.


In [15]:
# The raw submitted country against the conformed market list.
# left_only is exactly the master-data problem the country seed exists to solve.
result = w.overlap("stg.plan_submissions", "submitted_country",
                   "core.dim_market", "canonical_country")
display(result["summary"])
display(result["left_only"])

,left_distinct,right_distinct,matched,left_only,right_only
0,13,13,9,4,4


,unmatched_in_left
0,Brasil
1,Colombia
2,Deutschland
3,España


In [16]:
# And the same keys after harmonisation - this is what a clean join looks like.
result = w.overlap("stg.plan_submissions", "canonical_country",
                   "core.dim_market", "canonical_country")
display(result["summary"])

,left_distinct,right_distinct,matched,left_only,right_only
0,13,13,13,0,0


---

## 5. Before a QUALITY CHECK

Every check in this project is written in the same three steps, and the third is a
judgement rather than a query.

1. **The condition is written as a `SELECT` that returns the bad rows.** Returning
   nothing today is the check passing, not the check being wrong.
2. **They are counted, and the denominator is named.** `failure_count` and `denominator`
   are columns in every check here, because 3 bad rows out of 292 and 3 out of 3 are
   different situations.
3. **`blocking` is decided.** Does the defect make the number *wrong*, or the dataset
   *incomplete*? Wrong stops the build. Incomplete quarantines the row and warns.


In [17]:
# Step 1 - the condition, as a query that returns the offending rows.
display(w.q("""
    SELECT submission_batch, submitted_country, plan_month, plan_net_sales_local
    FROM raw.plan_submissions
    WHERE plan_net_sales_local < 0
"""))

,submission_batch,submitted_country,plan_month,plan_net_sales_local
0,FY2023-PLAN-V1,Belgium,2023-05-01,-2685.19


In [18]:
# Step 2 - the shape every check in this project is written in.
display(w.q("""
    SELECT 'plan_target_not_negative' AS check_name,
           COUNT(*) AS failure_count,
           (SELECT COUNT(*) FROM raw.plan_submissions) AS denominator,
           0 AS blocking
    FROM raw.plan_submissions WHERE plan_net_sales_local < 0
"""))

# Step 3 is the decision: a negative target makes the plan incomplete, not wrong,
# so blocking = 0 and the row goes to qa.plan_rejects instead of stopping the build.
display(w.peek("qa.plan_rejects"))

,check_name,failure_count,denominator,blocking
0,plan_target_not_negative,1,292,0


,submission_batch,submitted_country,plan_month,currency_code,plan_net_sales_local,reject_reason
0,FY2023-PLAN-V1,Belgium,2023-05-01,EUR,-2685.19,Plan value is negative
1,FY2023-PLAN-V1,France,2023-08-01,EUR,NaN,Plan value is blank
2,FY2023-PLAN-V1,Japan,2023-11-01,JPY,NaN,Plan value is blank
3,FY2024-PLAN-V1,Australia,2024-02-01,AUD,NaN,Plan value is blank
4,FY2024-PLAN-V1,United Kingdom,2024-09-01,GBP,NaN,Plan value is blank


---

## 6. Before a RECONCILIATION line

A reconciliation is not a test. It is a statement that a total survived a step, carrying
a written expectation of what should have happened to it.

Each line needs three things: the number *before*, the number *after*, and a sentence
saying whether the two are supposed to be equal. `tie()` supplies the first two; the
sentence is the `expectation` column, and it is written by hand.


In [19]:
# Staging may only type and rename, so this must tie exactly.
display(w.tie("gross sales, stg -> core",
              "SELECT SUM(sale_price) FROM stg.order_items",
              "SELECT SUM(gross_sales_value) FROM core.fact_order_item"))

,check,left,right,delta,verdict
0,"gross sales, stg -> core",10796335.7,10796335.7,0.0,ties


In [20]:
# Accepted plan rows must equal submitted rows minus the quarantine.
display(w.tie("plan rows, source -> core",
              "SELECT COUNT(*) - (SELECT COUNT(*) FROM qa.plan_rejects)\n               FROM raw.plan_submissions",
              "SELECT COUNT(*) FROM core.fact_plan"))

,check,left,right,delta,verdict
0,"plan rows, source -> core",287.0,287.0,0.0,ties


In [21]:
# The finished statement these lines become.
display(w.q("""
    SELECT stream, stage, entity, row_count, ROUND(value_usd, 2) AS value_usd,
           row_delta, expectation
    FROM qa.reconciliation_statement ORDER BY stream, step_order
"""))

,stream,stage,entity,row_count,value_usd,row_delta,expectation
0,Plan,Source file,regional_plan_submissions.csv,292,NaN,<NA>,Submitted in local currency - no USD value yet
1,Plan,stg,stg.plan_submissions,292,8078399.98,0,Harmonised and FX-converted - row count must equal source
2,Plan,qa,qa.plan_rejects,5,NaN,-287,"Quarantined rows, reported back to the submitting region"
3,Plan,core,core.fact_plan,287,8081299.98,282,Accepted rows only - source rows minus rejects
4,Sales,Source file,order_items.csv,180862,10796335.70,<NA>,Baseline: rows and gross value as delivered
5,Sales,stg,stg.order_items,180862,10796335.70,0,Typing and renaming only - must equal source
6,Sales,core,core.fact_order_item,180862,10796335.70,0,Joins to product and customer - must not drop rows
7,Sales,mart,mart.actual_country_month,135749,10796335.70,-45113,Aggregated to country x month - gross value must still tie


---

## 7. Reading the production code

The logic behind a number is read from the file rather than described from memory.
[`docs/code_map.md`](../docs/code_map.md) indexes every object in the warehouse to the
file and line that builds it; the two verbs below print the statement itself without
leaving the notebook.


In [22]:
display(w.owns("mart.price_volume_mix"))

,object,file,line,statement_lines
0,mart.price_volume_mix,sql/duckdb/08_commercial_planning.sql,235,93


In [23]:
w.source("core.dim_market")

--- sql/duckdb/02_core_dimensions.sql - 10 lines ----------------------
CREATE OR REPLACE TABLE core.dim_market AS
SELECT
    ROW_NUMBER() OVER (ORDER BY canonical_country)::INTEGER AS market_key,
    canonical_country,
    region,
    sub_region,
    COUNT(*) AS source_spellings_mapped
FROM core.dim_country
WHERE canonical_country <> 'Unknown'
GROUP BY canonical_country, region, sub_region;


In [24]:
# Long statements take a line limit.
w.source("mart.plan_vs_actual_monthly", 25)

--- sql/duckdb/08_commercial_planning.sql - 30 lines ----------------------
CREATE OR REPLACE TABLE mart.plan_vs_actual_monthly AS
SELECT
    COALESCE(a.canonical_country, p.canonical_country) AS canonical_country,
    COALESCE(a.region, p.region) AS region,
    COALESCE(a.month_start, p.plan_month) AS month_start,
    a.net_sales_value AS actual_net_sales_usd,
    a.net_units AS actual_units,
    p.plan_net_sales_usd,
    p.plan_units,
    a.net_sales_value - p.plan_net_sales_usd AS variance_usd,
    CASE WHEN p.plan_net_sales_usd > 0
         THEN (a.net_sales_value - p.plan_net_sales_usd) / p.plan_net_sales_usd END AS variance_rate,
    CASE WHEN p.plan_net_sales_usd > 0
         THEN a.net_sales_value / p.plan_net_sales_usd END AS achievement_rate,
    COALESCE(a.is_complete_month, 0) AS is_complete_month,
    CASE WHEN COALESCE(a.month_start, p.plan_month)
              BETWEEN (SELECT MIN(plan_month) FROM core.fact_plan)
                  AND (SELECT MAX(plan_month) FROM core.fac

In [25]:
# And the same for a Python function.
w.source_py("advanced_analytics.py", "rfm_segmentation", 12)

--- src/advanced_analytics.py - def rfm_segmentation - 118 lines ----------------
def rfm_segmentation(
    connection: duckdb.DuckDBPyConnection, output_dir: Path, figure_dir: Path
) -> dict[str, object]:
    customers = connection.execute(
        """
        SELECT customer_id, recency_days, order_count, net_sales_value,
               net_profit_value, observed_return_rate, country, acquisition_source
        FROM mart.customer_360
        WHERE net_sales_value > 0 AND order_count > 0
        """
    ).fetchdf()
    model_data = pd.DataFrame(
... 106 more lines, in the file


---

## 8. Scratch

Working space. Nothing below this point is read by the build, validated, or regenerated.


In [26]:
# w.q("""
#     SELECT ...
# """)

In [27]:
# 

In [28]:
w.close()